# Time Cell Minimal Loop (Train → Evaluate → Plot)
This notebook is **self-contained** (no `torchvision` dependency) and is designed to run a minimal closed-loop pipeline:
1) Load variable-length trajectory folders (pkl + images)
2) Train a Visual-Temporal LSTM model
3) Evaluate (MSE/R²) and plot predicted vs GT trajectory

**You only need to set `DATA_ROOT_DIR` to your dataset path.**

In [1]:
# [Cell 1] Imports & Environment Setup
%load_ext autoreload
%autoreload 2

import os
import torch
import numpy as np
from torch.utils.data import DataLoader

# --- 导入本地模块 (核心功能模块化) ---
# 1. 数据集
from dataset import VariableLengthTrajectoryDataset, collate_variable_length
# 2. 模型 (请确保 visualNet.py 已更新为 Optimized 版本)
from model.visualNet import VisualTemporalNet_Optimized 
# 3. 评估指标工具
from model.metrics import NMI_Evaluator
# 4. NMI 画图与分析工具
from model import plotcell, evalcell, evaltraj

# --- 硬件配置 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# --- 数据集加载 (保持原有的加载逻辑) ---
# 请修改为你实际的数据路径
DATA_ROOT_DIR = "/media/zhen/Data/Datasets/nomad_data/go_stanford" 

if not os.path.exists(DATA_ROOT_DIR):
    print(f"⚠️ Warning: Path {DATA_ROOT_DIR} does not exist. Please check config.")
else:
    # 加载数据集 (保留原有参数)
    train_dataset = VariableLengthTrajectoryDataset(DATA_ROOT_DIR, min_len=10)
    print(f"✅ Dataset loaded: {len(train_dataset)} trajectories found.")

✅ Using device: cuda
✅ Found 3696 trajectories in /media/zhen/Data/Datasets/nomad_data/go_stanford
✅ Dataset loaded: 3696 trajectories found.


/media/zhen/Data/cogMap/biw/scale_ablation/model/evalcell.py:156: SyntaxWarning: invalid escape sequence '\s'
  plt.plot(smoothed, label=f"Smoothed ($\sigma={s}$)",


In [2]:
# [Cell 2] Global Configuration

# --- 1. 实验保存路径 ---
RESULTS_DIR = "./eval_results_minimal"
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- 2. 训练超参数 ---
TRAIN_CONFIG = {
    'epochs': 30,           # 建议 30+ 以观察收敛
    'lr': 1e-3,             # 学习率
    'batch_size': 4,        # 显存允许可调大
    'device': device
}

# --- 3. 模型结构参数 ---
# 确保 feature_dim 与你 dataset 输出的维度一致 (通常是 x,y,yaw = 3)
MODEL_CONFIG = {
    'feature_dim': 3,       
    'hidden_dim': 128,      
    'visual_dim': 256,
    'output_dim': 2         # 预测 x, y
}


In [3]:
# [Cell 3] Define Separated Training and Evaluation Pipelines

def get_exp_dir(exp_name):
    """辅助函数：获取实验目录"""
    path = os.path.join(RESULTS_DIR, exp_name)
    os.makedirs(path, exist_ok=True)
    return path

def run_training(exp_name, dataset, ablation_config, train_cfg=TRAIN_CONFIG, model_cfg=MODEL_CONFIG):
    """
    【阶段一】训练管道：构建模型 -> 训练 -> 保存权重
    """
    save_dir = get_exp_dir(exp_name)
    print(f"\n{'='*10} [TRAIN] START: {exp_name} {'='*10}")
    
    # 1. 初始化模型
    print(f"Building model architecture: {ablation_config}")
    model = VisualTemporalNet_Optimized(
        feature_dim=model_cfg['feature_dim'],
        hidden_dim=model_cfg['hidden_dim'],
        visual_dim=model_cfg['visual_dim'],
        output_dim=model_cfg['output_dim'],
        ablation_config=ablation_config
    ).to(train_cfg['device'])
    
    optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg['lr'])
    criterion = torch.nn.MSELoss(reduction='none') 
    
    # 2. 训练循环
    loader = DataLoader(dataset, batch_size=train_cfg['batch_size'], 
                        shuffle=True, collate_fn=collate_variable_length)
    
    model.train()
    for epoch in range(train_cfg['epochs']):
        total_loss = 0
        batch_count = 0
        
        # [Fix] Ignore endpoint targets, extract sequence targets from feats
        for feats, imgs, _, mask in loader:
            if feats is None: continue 
            
            # 构建序列 targets [B, T, 2]
            targets_seq = feats[:, :, :2].clone()
            
            feats = feats.to(train_cfg['device'])
            imgs = imgs.to(train_cfg['device'])
            targets_seq = targets_seq.to(train_cfg['device'])
            mask = mask.to(train_cfg['device'])
            
            optimizer.zero_grad()
            preds, _ = model(feats, imgs, mask)
            
            # Masked Loss
            raw_loss = criterion(preds, targets_seq)
            masked_loss = (raw_loss.mean(dim=-1) * mask).sum() / (mask.sum() + 1e-6)
            
            masked_loss.backward()
            optimizer.step()
            
            total_loss += masked_loss.item()
            batch_count += 1
            
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{train_cfg['epochs']} | Avg Loss: {total_loss/(batch_count+1e-6):.6f}")

    # 3. 保存模型权重 (这是分离的关键！)
    model_path = os.path.join(save_dir, "model_final.pth")
    torch.save(model.state_dict(), model_path)
    print(f"✅ Model saved to: {model_path}")
    
    return model_path


def run_evaluation(exp_name, dataset, ablation_config, model_cfg=MODEL_CONFIG, device=device):
    """
    【阶段二】评估管道：加载权重 -> 计算指标 -> 画图
    此函数可以反复运行，无需重新训练。
    """
    save_dir = get_exp_dir(exp_name)
    model_path = os.path.join(save_dir, "model_final.pth")
    
    print(f"\n{'='*10} [EVAL] START: {exp_name} {'='*10}")
    
    if not os.path.exists(model_path):
        print(f"❌ Error: Model file not found at {model_path}. Please run training first.")
        return None

    # 1. 重建模型结构并加载权重
    print(f"Loading model from {model_path}...")
    model = VisualTemporalNet_Optimized(
        feature_dim=model_cfg['feature_dim'],
        hidden_dim=model_cfg['hidden_dim'],
        visual_dim=model_cfg['visual_dim'],
        output_dim=model_cfg['output_dim'],
        ablation_config=ablation_config
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_variable_length)

    # 2. 基础指标计算 (RMSE, NRMSE)
    print("Computing metrics...")
    all_preds, all_targets = [], []
    with torch.no_grad():
        for feats, imgs, _, mask in loader:
            if feats is None: continue
            
            # [Fix] Extract Sequence Targets
            targets_seq = feats[:, :, :2].clone()
            
            feats, imgs, targets_seq, mask = feats.to(device), imgs.to(device), targets_seq.to(device), mask.to(device)
            preds, _ = model(feats, imgs, mask)
            
            mask_bool = mask.bool().cpu().numpy().flatten()
            p = preds.cpu().numpy().reshape(-1, 2)
            t = targets_seq.cpu().numpy().reshape(-1, 2)
            
            all_preds.append(p[mask_bool])
            all_targets.append(t[mask_bool])
            
    if len(all_preds) > 0:
        flat_preds = np.concatenate(all_preds, axis=0)
        flat_targets = np.concatenate(all_targets, axis=0)
        metrics = NMI_Evaluator.calculate_metrics(flat_targets, flat_preds)
        NMI_Evaluator.print_summary(metrics, prefix=exp_name)
    else:
        return None

    # 3. 高级评估与绘图 (NMI Pipeline)
    print(f"Generating plots in {save_dir}...")
    
    # (A) Time Cell Analysis
    cell_results = plotcell.analyze_time_cells(model, loader, device, save_dir=save_dir)
    
    if cell_results:
        # (B) Inference Ablation
        evalcell.evaluate_time_scales_and_extensions(model, loader, cell_results, device, save_dir=save_dir)
        # (C) Trajectory Correlation
        evaltraj.evaluate_timecell_trajectory_correlation(model, loader, cell_results, device, save_dir=save_dir)
    
    # (D) Final Trajectory Visualization
    evaltraj.evaluate_and_visualize(model, loader, device, model_path="", save_dir=save_dir, save_prefix="final", plot=True)
    
    print(f"✅ Evaluation finished for {exp_name}")
    return metrics

In [4]:
# [Cell 4] Step A: Training Phase (Run this ONCE)

# 1. 完整模型
run_training("Exp_Full_Fusion", train_dataset, ablation_config={'use_visual': True, 'use_kinematics': True})

# 2. 仅视觉
run_training("Exp_Visual_Only", train_dataset, ablation_config={'use_visual': True, 'use_kinematics': False})

# 3. 仅动力学
run_training("Exp_Kinematics_Only", train_dataset, ablation_config={'use_visual': False, 'use_kinematics': True})


========== [TRAIN] START: Exp_Full_Fusion ==========
Building model architecture: {'use_visual': True, 'use_kinematics': True}


/media/zhen/Share/anaconda3/envs/cogmap_torchtem/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/media/zhen/Share/anaconda3/envs/cogmap_torchtem/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


  Epoch 5/30 | Avg Loss: 33.110574
  Epoch 10/30 | Avg Loss: 21.807229
  Epoch 15/30 | Avg Loss: 21.169102
  Epoch 20/30 | Avg Loss: 12.585807
  Epoch 25/30 | Avg Loss: 14.229337
  Epoch 30/30 | Avg Loss: 13.192017
✅ Model saved to: ./eval_results_minimal/Exp_Full_Fusion/model_final.pth

========== [TRAIN] START: Exp_Visual_Only ==========
Building model architecture: {'use_visual': True, 'use_kinematics': False}
  Epoch 5/30 | Avg Loss: 424.631629
  Epoch 10/30 | Avg Loss: 421.442772
  Epoch 15/30 | Avg Loss: 414.690165
  Epoch 20/30 | Avg Loss: 422.295004
  Epoch 25/30 | Avg Loss: 424.448129
  Epoch 30/30 | Avg Loss: 426.282342
✅ Model saved to: ./eval_results_minimal/Exp_Visual_Only/model_final.pth

========== [TRAIN] START: Exp_Kinematics_Only ==========
Building model architecture: {'use_visual': False, 'use_kinematics': True}
  Epoch 5/30 | Avg Loss: 49.127966
  Epoch 10/30 | Avg Loss: 32.253793
  Epoch 15/30 | Avg Loss: 30.905000
  Epoch 20/30 | Avg Loss: 24.909207
  Epoch 25/30

'./eval_results_minimal/Exp_Kinematics_Only/model_final.pth'

In [5]:
# [Cell 5] Step B: Evaluation Phase (Run repeatedly to tweak plots)

metrics_full = run_evaluation("Exp_Full_Fusion", train_dataset, ablation_config={'use_visual': True, 'use_kinematics': True})
metrics_vis = run_evaluation("Exp_Visual_Only", train_dataset, ablation_config={'use_visual': True, 'use_kinematics': False})
metrics_kin = run_evaluation("Exp_Kinematics_Only", train_dataset, ablation_config={'use_visual': False, 'use_kinematics': True})

print("\n--- Final Comparison ---")
if metrics_full: print(f"Full Fusion RMSE: {metrics_full['RMSE']:.4f}")


========== [EVAL] START: Exp_Full_Fusion ==========
Loading model from ./eval_results_minimal/Exp_Full_Fusion/model_final.pth...
Computing metrics...
[Exp_Full_Fusion Summary] RMSE: 3.6101 | NRMSE: 0.1692 | R²: 0.9711
    > Detail: RMSE_x=3.4540, RMSE_y=3.7596
Generating plots in ./eval_results_minimal/Exp_Full_Fusion...
Hidden states padded to (3696, 849, 128)
Preparing data for PCA...
Running ablation analysis...


/media/zhen/Data/cogMap/biw/scale_ablation/model/evalcell.py:189: RuntimeWarning: Mean of empty slice
  avg_cross_corr = np.nanmean(corr_cross_traj)


Ablation evaluation completed.
Computing synchronized hidden states and trajectories for correlation...
Starting trajectory evaluation...
Evaluation Results: MSE=13.032487, R²=0.9711
Trajectory plot saved.
✅ Evaluation finished for Exp_Full_Fusion

========== [EVAL] START: Exp_Visual_Only ==========
Loading model from ./eval_results_minimal/Exp_Visual_Only/model_final.pth...


/media/zhen/Share/anaconda3/envs/cogmap_torchtem/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/media/zhen/Share/anaconda3/envs/cogmap_torchtem/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Computing metrics...
[Exp_Visual_Only Summary] RMSE: 21.3576 | NRMSE: 1.0009 | R²: -0.0004
    > Detail: RMSE_x=22.1237, RMSE_y=20.5630
Generating plots in ./eval_results_minimal/Exp_Visual_Only...
Hidden states padded to (3696, 849, 128)
Preparing data for PCA...
Running ablation analysis...


...

Ablation evaluation completed.
Computing synchronized hidden states and trajectories for correlation...
Starting trajectory evaluation...
Evaluation Results: MSE=456.147461, R²=-0.0004
Trajectory plot saved.
✅ Evaluation finished for Exp_Visual_Only

========== [EVAL] START: Exp_Kinematics_Only ==========
Loading model from ./eval_results_minimal/Exp_Kinematics_Only/model_final.pth...


/media/zhen/Share/anaconda3/envs/cogmap_torchtem/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/media/zhen/Share/anaconda3/envs/cogmap_torchtem/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Computing metrics...
[Exp_Kinematics_Only Summary] RMSE: 4.6983 | NRMSE: 0.2202 | R²: 0.9510
    > Detail: RMSE_x=4.4782, RMSE_y=4.9085
Generating plots in ./eval_results_minimal/Exp_Kinematics_Only...
Hidden states padded to (3696, 849, 128)
Preparing data for PCA...
Running ablation analysis...


/media/zhen/Data/cogMap/biw/scale_ablation/model/evalcell.py:189: RuntimeWarning: Mean of empty slice
  avg_cross_corr = np.nanmean(corr_cross_traj)


Ablation evaluation completed.
Computing synchronized hidden states and trajectories for correlation...
Starting trajectory evaluation...
Evaluation Results: MSE=22.073914, R²=0.9510
Trajectory plot saved.
✅ Evaluation finished for Exp_Kinematics_Only

--- Final Comparison ---
Full Fusion RMSE: 3.6101
